# Lab 2.1 &mdash; Chain-of-Thought, Built as a Chain

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Compose your first LCEL chain: <code>prompt | model | parser</code>
- Build two arms that differ only in their prompt, and nothing else
- Run a whole eval set in one call with <code>.batch()</code>
- Measure what &ldquo;think step by step&rdquo; is actually worth on your task

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 2 labs work one case: payment exceptions on a small
> synthetic ledger. Module 1 built the agent loop; Module 2 is about what goes on
> inside one turn of it.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 2 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

**Chain-of-Thought** asks the model to show its working before it answers. It usually helps on
multi-step tasks and usually costs latency, and *how much* of each is a property of your task,
not a fact about LLMs. So measure it.

The mechanism you use to measure it is worth as much as the answer. **LCEL** &mdash; LangChain
Expression Language &mdash; composes a prompt, a model and a parser into one runnable with `|`:

```python
chain = prompt | model | StrOutputParser()
chain.invoke({"case": ...})        # one
chain.batch([{...}, {...}, ...])   # many, concurrently
```

Everything downstream in this course is built this way, and `.batch()` is what makes an eval set
practical instead of a coffee break.

## Section 1 &mdash; Your first chain

Three parts, one pipe. `ChatPromptTemplate` turns variables into messages; the model answers;
`StrOutputParser` pulls `.content` out so the chain returns a plain string.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import AIMessage

DIRECT = ("You are a payments operations analyst. Answer with the single next action, "
          "and nothing else.")

def output_parser():
    """The last stage of the chain: what turns the AIMessage into a plain string?"""
    return BLANK                      # TODO: which parser, INSTANTIATED -- note the ()


def build_chain(system: str):
    """prompt | model | parser -- the three-part chain every later lab uses."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", system),
        ("human", "PAYMENT: {payment}\nPOLICY: {policy}\n\nWhat must happen next?"),
    ])
    return prompt | get_llm() | output_parser()

In [ ]:
# --- Self-check: Section 1   (the prompt half is pure -- no model call)
_p = ChatPromptTemplate.from_messages([
    ("system", DIRECT),
    ("human", "PAYMENT: {payment}\nPOLICY: {policy}\n\nWhat must happen next?"),
])

check("the template declares both variables",
      lambda: set(_p.input_variables) == {"payment", "policy"})
check("it renders to a system message and a human message",
      lambda: [m.type for m in _p.invoke({"payment": "p", "policy": "q"}).messages]
              == ["system", "human"])
check("the variables are substituted, not left as braces",
      lambda: "PAYMENT: p" in _p.invoke({"payment": "p", "policy": "q"}).messages[1].content)

check("the chain ends in a string parser",
      lambda: isinstance(output_parser(), StrOutputParser),
      "it must be StrOutputParser() -- an INSTANCE; passing the class silently breaks the pipe")
check("the parser turns an AIMessage into a plain string",
      lambda: output_parser().invoke(AIMessage("hello")) == "hello",
      "that is all StrOutputParser does, and it is why the chain returns str not AIMessage")

## Section 2 &mdash; Two arms, one difference

Both arms use the **same** `build_chain`. The only thing that changes is the system prompt. That
is what makes the comparison worth anything: if you also changed the model, the temperature or
the question, you would learn nothing from the result.

Write the Chain-of-Thought prompt. It has to ask for the working *and* keep the final answer
findable, or you cannot score it.

In [ ]:
ANSWER_MARKER = "ACTION:"

# TODO: replace this string with your Chain-of-Thought system prompt. It must ask for the
# reasoning first and finish with a line beginning ANSWER_MARKER, or you cannot score it.
COT = "BLANK"

In [ ]:
def final_action(text: str) -> str:
    """The answer, separated from the working.

    With the marker: everything after the LAST marker line.
    Without it: the whole text, which is what the direct arm produces.
    """
    if ANSWER_MARKER in text:
        return text.rsplit(ANSWER_MARKER, 1)[1].strip()
    return text.strip()

In [ ]:
# --- Self-check: Section 2   (string handling only -- no model call)
def _cot():
    # A blank inside a string is not a blank -- it is the literal word, and reading it can
    # never raise. So raise it by hand, or an untouched lab shows [FAIL] instead of [TODO].
    if not isinstance(COT, str) or COT.strip() == "BLANK":
        raise NameError("COT is not written yet")
    return COT

check("the CoT prompt asks for reasoning",
      lambda: any(w in _cot().lower() for w in ("step by step", "step-by-step", "work through")),
      "if it does not ask for the working, it is not Chain-of-Thought")
check("the CoT prompt names the answer marker",
      lambda: ANSWER_MARKER in _cot(),
      "you have to be able to find the answer inside the working, or you cannot score it")
check("both arms are still the same analyst",
      lambda: "payments operations analyst" in _cot(),
      "change one thing between arms -- the reasoning instruction -- and nothing else")
check("final_action strips the working away",
      lambda: final_action("thinking...\nACTION: Escalate to Treasury.") == "Escalate to Treasury.")
check("final_action takes the LAST marker",
      lambda: final_action("ACTION: wrong\nmore\nACTION: right") == "right",
      "models sometimes restate the format before using it")
check("an unmarked answer survives unchanged",
      lambda: final_action("Escalate to Treasury.") == "Escalate to Treasury.")

## Section 3 &mdash; The eval set, and `.batch()`

Five cases, each with the terms a correct action must mention. Crude on purpose &mdash; it has to be
something you can defend and re-run, not something you have to read.

`.batch()` sends the whole list at once and returns the answers in order. It is the difference
between an eval set you run every time and one you ran once.

In [ ]:
CASES = [
    {"ref": "PMT-1003", "must_contain": ["treasury"]},
    {"ref": "PMT-1005", "must_contain": ["compliance"]},
    {"ref": "PMT-1002", "must_contain": ["retry"]},
    {"ref": "PMT-1004", "must_contain": ["originator", "r04"]},
    {"ref": "PMT-1001", "must_contain": ["settled", "no action", "none"]},
]

def inputs_for(case: dict) -> dict:
    """The variables one case supplies to the chain."""
    rec = LEDGER[case["ref"]]
    return {"payment": json.dumps({"ref": case["ref"], **rec}),
            "policy": POLICY.get(rec["reason_code"], "no policy applies")}


def scores(answers: list[str]) -> list[bool]:
    """One bool per case: did the final action mention any required term?"""
    out = []
    for case, answer in zip(CASES, answers):
        action = final_action(answer).lower()
        out.append(BLANK)             # TODO: any required term present, or all of them?
    return out


def run_arm(system: str) -> dict:
    """Run every case through one arm, in a single batched call."""
    chain = build_chain(system)
    t0 = time.time()
    answers = chain.batch([inputs_for(c) for c in CASES])
    return {"answers": answers, "scores": scores(answers), "seconds": time.time() - t0}

In [ ]:
# --- Self-check: Section 3   (the scorer, on canned answers -- no model call)
_canned = ["Escalate to Treasury for approval.",
           "Hold; Compliance decides.",
           "Retry once after 24h.",
           "Do something vague.",
           "Already settled, no action."]

check("every case names a payment that exists",
      lambda: all(c["ref"] in LEDGER for c in CASES))
check("the chain inputs carry the payment and the policy",
      lambda: set(inputs_for(CASES[0])) == {"payment", "policy"})
check("a case with no reason code still gets a policy string",
      lambda: isinstance(inputs_for(CASES[4])["policy"], str),
      "PMT-1001 is settled -- the chain must still receive something for {policy}")
check("the scorer accepts a correct answer", lambda: scores(_canned)[0] is True)
check("the scorer rejects a vague one",      lambda: scores(_canned)[3] is False)
check("either term is enough",
      lambda: scores(["x", "x", "x", "Send it back with code R04.", "x"])[3] is True,
      'must_contain is a list of alternatives -- "any", not "all"')
check("the scorer reads only the final action",
      lambda: scores(["I considered Treasury but ACTION: hold for Compliance.",
                      "x", "x", "x", "x"])[0] is False,
      "reasoning that mentions the right word is not the same as answering it")

## Run it for real

Both arms, five cases each, two batched calls. Watch the per-case table rather than the totals.

In [ ]:
if llm_ready():
    def _compare():
        direct = run_arm(DIRECT)
        cot    = run_arm(COT)
        print("  case       direct   chain-of-thought")
        for c, a, b in zip(CASES, direct["scores"], cot["scores"]):
            f = lambda x: "pass" if x else "FAIL"
            print(f"  {c['ref']}   {f(a):8} {f(b)}")
        n = len(CASES)
        print(f"\n  direct           {sum(direct['scores'])}/{n}   {direct['seconds']:.1f}s")
        print(f"  chain-of-thought {sum(cot['scores'])}/{n}   {cot['seconds']:.1f}s")
        print("\n--- one CoT answer in full ---\n")
        print(cot["answers"][0][:700])
        return {"direct": direct, "cot": cot}
    ARMS = guard(_compare)

### Read it

Three things to look for.

1. **Which cases moved.** CoT rarely helps uniformly. It tends to earn its keep exactly where the
   answer needs two facts joined &mdash; the reason code *and* what policy says about it &mdash; and to
   do nothing at all where one lookup was enough.
2. **What the working looks like.** Read the full answer printed at the end. The model states the
   status, then the code, then the policy. That ordering is the whole mechanism: each step lands
   in the context before the next one needs it.
3. **The marker earned its place.** Without `ACTION:` you would be scoring the reasoning as well
   as the answer, and CoT would appear to win simply by mentioning more words. The
   `final_action` self-check above encodes exactly that trap.

Note also what `.batch()` did: five cases went out together rather than one after another. The
same eval set run serially takes several times as long, which is usually the difference between
a suite people run and one they skip.

In [ ]:
score()

## Your turn

1. Add a sixth case whose correct action needs **three** facts joined, and see whether the gap
   between the arms widens. That is the shape of task where CoT pays.
2. Replace `StrOutputParser()` with `with_structured_output` returning a small model that has
   `reasoning` and `action` fields. You no longer need `ANSWER_MARKER` or `final_action` at all
   &mdash; which of the two designs would you rather maintain?
3. Run `run_arm(COT)` three times. The scores will not be identical. Decide how many runs your
   eval set needs before you would let it gate a release, and write the number down.